In [10]:

valid_pickups = df.dropna(subset=["pickup"]).copy()

valid_pickups["day_of_week"] = (
    valid_pickups["pickup"].dt.day_name()
)

revenue_by_day = (
    valid_pickups.groupby("day_of_week")["total"]
    .sum()
    .reindex([
        "Monday", "Tuesday", "Wednesday", "Thursday",
        "Friday", "Saturday", "Sunday"
    ])
)

revenue_by_day = revenue_by_day.dropna().sort_values(ascending=False)

best_revenue_day = revenue_by_day.index[0]
best_revenue = revenue_by_day.iloc[0]

print("Revenue by day of the week:")
print(revenue_by_day.round(2))

print(
    f"\nIn plain English, {best_revenue_day} brings in the most "
    f"recorded revenue, totaling ${best_revenue:,.2f}."
)


Revenue by day of the week:
day_of_week
Friday       20804.21
Wednesday    18481.79
Saturday     18301.77
Thursday     17106.07
Tuesday      15583.30
Sunday       15421.16
Monday       13426.67
Name: total, dtype: float64

In plain English, Friday brings in the most recorded revenue, totaling $20,804.21.



## 8. Which day of the week brings in the most revenue?

I will use each ride's pickup date to group rides by day of the week and add their total values.


In [9]:

valid_distance = df[df["distance"] > 0].copy()

valid_distance["fare_per_mile"] = (
    valid_distance["fare"] / valid_distance["distance"]
)

long_rides = valid_distance[valid_distance["distance"] > 10]
short_rides = valid_distance[valid_distance["distance"] < 2]

long_fare_per_mile = long_rides["fare_per_mile"].mean()
short_fare_per_mile = short_rides["fare_per_mile"].mean()

print(f"Rides over 10 miles: {len(long_rides):,}")
print(f"Average fare per mile: ${long_fare_per_mile:.2f}")

print(f"\nRides under 2 miles: {len(short_rides):,}")
print(f"Average fare per mile: ${short_fare_per_mile:.2f}")

if pd.notna(long_fare_per_mile) and pd.notna(short_fare_per_mile):
    if long_fare_per_mile > short_fare_per_mile:
        comparison = "Long rides have the higher average fare per mile."
    elif long_fare_per_mile < short_fare_per_mile:
        comparison = "Short rides have the higher average fare per mile."
    else:
        comparison = "Both groups have the same average fare per mile."
else:
    comparison = "There is not enough valid data to compare both groups."

print(f"\nIn plain English, {comparison}")


Rides over 10 miles: 394
Average fare per mile: $3.09

Rides under 2 miles: 3,703
Average fare per mile: $7.60

In plain English, Short rides have the higher average fare per mile.



## 7. How does the fare per mile compare for rides over 10 miles and rides under 2 miles?

I will calculate fare per mile for each ride, then compare the average for long and short rides.


In [8]:

top_zones = df["pickup_zone"].value_counts().head(5)

print("Top five pickup zones:")
print(top_zones)

print(
    f"In plain English, {top_zones.index[0]} is the busiest pickup zone "
    f"with {top_zones.iloc[0]:,} rides."
)


Top five pickup zones:
pickup_zone
Midtown Center                  230
Upper East Side South           211
Penn Station/Madison Sq West    210
Clinton East                    208
Midtown East                    198
Name: count, dtype: int64
In plain English, Midtown Center is the busiest pickup zone with 230 rides.



## 6. What are the top five pickup zones by number of rides?

I will count the rides in each pickup zone and display the five busiest zones.


In [7]:

known_boroughs = df.dropna(
    subset=["pickup_borough", "dropoff_borough"]
)

different_borough = (
    known_boroughs["pickup_borough"]
    != known_boroughs["dropoff_borough"]
)

different_borough_count = different_borough.sum()
known_borough_count = len(known_boroughs)

different_borough_share = (
    different_borough_count / known_borough_count * 100
    if known_borough_count > 0
    else 0
)

print(f"Rides with both boroughs known: {known_borough_count:,}")
print(f"Rides ending in a different borough: {different_borough_count:,}")
print(f"Share: {different_borough_share:.2f}%")

print(
    f"In plain English, {different_borough_share:.2f}% of rides "
    "with known pickup and dropoff boroughs end in a different borough."
)


Rides with both boroughs known: 6,383
Rides ending in a different borough: 801
Share: 12.55%
In plain English, 12.55% of rides with known pickup and dropoff boroughs end in a different borough.



## 5. What share of rides end in a different borough from the one they started in?

I will compare the pickup and dropoff boroughs for rides where both boroughs are known.


In [6]:

pickup_hours = df.dropna(subset=["pickup"]).copy()

pickup_hours["hour"] = pickup_hours["pickup"].dt.hour

rides_by_hour = pickup_hours["hour"].value_counts().sort_index()

busiest_hour = rides_by_hour.idxmax()
busiest_hour_count = rides_by_hour.max()

print("Rides by pickup hour:")
print(rides_by_hour)

print(
    f"\nIn plain English, the busiest pickup hour is {busiest_hour}:00, "
    f"with {busiest_hour_count:,} rides."
)


Rides by pickup hour:
hour
0     205
1     110
2     101
3      67
4      57
5      51
6     142
7     221
8     313
9     319
10    325
11    295
12    334
13    316
14    360
15    330
16    336
17    388
18    417
19    406
20    368
21    355
22    321
23    296
Name: count, dtype: int64

In plain English, the busiest pickup hour is 18:00, with 417 rides.



## 4. Which hour of the day has the most pickups?

I will extract the hour from each pickup time and count the rides in each hour.


In [5]:

valid_fares = df[df["fare"] > 0].copy()

valid_fares["tip_share"] = (
    valid_fares["tip"] / valid_fares["fare"]
)

payment_tip_share = (
    valid_fares[valid_fares["payment"].isin(["credit card", "cash"])]
    .groupby("payment")["tip_share"]
    .mean()
    * 100
)

print("Average tip as a share of fare:")
print(payment_tip_share.round(2).astype(str) + "%")

for payment, share in payment_tip_share.items():
    print(
        f"In plain English, {payment} payments have an average tip "
        f"equal to {share:.2f}% of the fare."
    )


Average tip as a share of fare:
payment
cash             0.0%
credit card    23.78%
Name: tip_share, dtype: str
In plain English, cash payments have an average tip equal to 0.00% of the fare.
In plain English, credit card payments have an average tip equal to 23.78% of the fare.



## 3. What is the average tip as a share of the fare for card payments and for cash?

I will calculate each ride's tip divided by its fare, then compare the average for each payment type.


In [4]:

borough_rides = df.groupby("pickup_borough").size().sort_values(
    ascending=False
)

borough_fares = df.groupby("pickup_borough")["fare"].mean().dropna().sort_values(
    ascending=False
)

most_rides_borough = borough_rides.index[0]
highest_fare_borough = borough_fares.index[0]

print("Rides by pickup borough:")
print(borough_rides)

print("\nAverage fare by pickup borough:")
print(borough_fares.round(2))

print(
    f"\nIn plain English, {most_rides_borough} has the most rides, "
    f"while {highest_fare_borough} has the highest average fare."
)


Rides by pickup borough:
pickup_borough
Manhattan    5268
Queens        657
Brooklyn      383
Bronx          99
dtype: int64

Average fare by pickup borough:
pickup_borough
Queens       24.93
Bronx        21.00
Brooklyn     16.52
Manhattan    11.15
Name: fare, dtype: float64

In plain English, Manhattan has the most rides, while Queens has the highest average fare.



## 2. Which pickup borough has the most rides, and which has the highest average fare?

I will group the rides by pickup borough and compare the ride counts and average fares.


In [3]:

ride_count = len(df)
total_revenue = df["total"].sum()

print(f"Total rides: {ride_count:,}")
print(f"Total revenue: ${total_revenue:,.2f}")
print(
    f"In plain English, the dataset contains {ride_count:,} rides "
    f"with total recorded revenue of ${total_revenue:,.2f}."
)


Total rides: 6,433
Total revenue: $119,124.97
In plain English, the dataset contains 6,433 rides with total recorded revenue of $119,124.97.



## 1. How many rides are there, and what is the total revenue?

I will count the rows and add the values in the total column.


In [2]:

df["pickup"] = pd.to_datetime(df["pickup"], errors="coerce")
df["dropoff"] = pd.to_datetime(df["dropoff"], errors="coerce")

print("Pickup and dropoff times are ready for analysis.")



Pickup and dropoff times are ready for analysis.


In [1]:

import pandas as pd

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/taxis.csv"

df = pd.read_csv(url)

print("Dataset shape:", df.shape)
print(df.head())
print(df.columns.tolist())


Dataset shape: (6433, 14)
                pickup              dropoff  passengers  distance  fare   tip  \
0  2019-03-23 20:21:09  2019-03-23 20:27:24           1      1.60   7.0  2.15   
1  2019-03-04 16:11:55  2019-03-04 16:19:00           1      0.79   5.0  0.00   
2  2019-03-27 17:53:01  2019-03-27 18:00:25           1      1.37   7.5  2.36   
3  2019-03-10 01:23:59  2019-03-10 01:49:51           1      7.70  27.0  6.15   
4  2019-03-30 13:27:42  2019-03-30 13:37:14           3      2.16   9.0  1.10   

   tolls  total   color      payment            pickup_zone  \
0    0.0  12.95  yellow  credit card        Lenox Hill West   
1    0.0   9.30  yellow         cash  Upper West Side South   
2    0.0  14.16  yellow  credit card          Alphabet City   
3    0.0  36.95  yellow  credit card              Hudson Sq   
4    0.0  13.40  yellow  credit card           Midtown East   

            dropoff_zone pickup_borough dropoff_borough  
0    UN/Turtle Bay South      Manhattan       Manh


# Day 2: Exploring New York Taxi Rides

Today I am using Pandas to explore taxi ride data and answer eight business questions. I will calculate the results and explain what each result means in simple English.
